# Modulo 04 - File e Funzioni

---

Finora, tutti i dati che abbiamo usato nascevano e morivano dentro il notebook. In questo modulo imparerai a **leggere** e **scrivere file**, in modo che i tuoi programmi possano lavorare con dati reali (come la base clienti di una banca in formato CSV) e conservare risultati che continuano a esistere dopo la fine del programma. In seguito conosceremo le **funzioni**, che permettono di racchiudere un pezzo di codice, dargli un nome e riutilizzarlo tutte le volte che vogliamo, come in una simulazione di investimenti con vari scenari. Infine capiremo lo **scope** (ambito di visibilità), cioè dove ogni variabile esiste e per quanto tempo. File e funzioni sono presenti in praticamente ogni progetto di dati e di *software*, e sono la base per i moduli successivi.

Corso: Ready To Deploy

Creato da: [Enzo Schitini](https://www.linkedin.com/in/enzoschitini)

---

## Argomenti

| **Argomento** | Descrizione |
| --- | --- |
| 1. Lettura di file | Come aprire i file con `with` e `open()` e leggere il contenuto intero o riga per riga, applicato a una base clienti. |
| 2. Scrittura di file | Modalità di apertura, creazione, sovrascrittura e aggiunta di contenuto, oltre a filtrare dati da un file a un altro. |
| 3. Funzioni | Come definire funzioni, usare parametri e valori di ritorno, applicato a una simulazione di interesse composto. |
| 4. Scope | Dove esistono le variabili: scope locale, scope globale e il comportamento di `if` e `for`. |

---

## 1. Lettura di file

Nella vita reale, i dati raramente vengono digitati a mano nel codice: arrivano in **file** esportati da sistemi, fogli di calcolo o database. Saper aprire e leggere un file è il primo passo di qualsiasi analisi di dati.

### 1.1 Motivazione

Lavori nel team dati di una banca e hai ricevuto il file `banco.csv`, con informazioni su alcuni clienti. Il primo compito è semplice: **scoprire l'età media dei clienti**.

Un file **CSV** (*Comma-Separated Values*, ovvero "valori separati da virgola") è un file di testo in cui ogni riga rappresenta un record e le colonne sono separate da virgole. La prima riga, chiamata **intestazione**, contiene il nome delle colonne. Nel nostro file sono in inglese:

| Colonna | Significato |
| --- | --- |
| `age` | Età |
| `job` | Professione |
| `marital` | Stato civile |
| `education` | Istruzione |
| `default` | È insolvente? |
| `balance` | Saldo in conto |
| `housing` | Ha un mutuo per la casa? |
| `loan` | Ha un prestito personale? |

**Come possiamo portare il contenuto di questo file dentro Python e calcolare l'età media?**

### 1.2 Preparare il file di esempio

Prima di iniziare, dobbiamo far sì che il file `banco.csv` esista nella cartella in cui il notebook è in esecuzione. La cella sotto crea questo file. Non preoccuparti dei dettagli del codice per ora: usa esattamente ciò che impareremo nella sezione 2 (scrittura di file).

In [1]:
# Contenuto del file: un cliente per riga, colonne separate da virgola
conteudo_banco = '''age,job,marital,education,default,balance,housing,loan
30,unemployed,married,primary,no,1787,no,no
33,services,married,secondary,no,4789,yes,yes
35,management,single,tertiary,no,1350,yes,no
30,management,married,tertiary,no,1476,yes,yes
59,blue-collar,married,secondary,no,0,yes,no
35,management,single,tertiary,no,747,no,no
36,self-employed,married,tertiary,no,307,yes,no
39,technician,married,secondary,no,147,yes,no
41,entrepreneur,married,tertiary,no,221,yes,no
43,services,married,primary,no,-88,yes,yes
'''

# Creazione del file banco.csv (vedremo come funziona nella sezione 2)
with open('banco.csv', mode='w', encoding='utf-8') as arquivo:
    arquivo.write(conteudo_banco)

print('File banco.csv creato!')

Arquivo banco.csv criado!


> 💡 **Suggerimento:** su Google Colab, clicca sull'icona della **cartella** (📁) nella barra laterale sinistra per vedere i file della sessione. Il `banco.csv` apparirà lì, e puoi persino scaricarlo. Ricorda che questi file sono **temporanei** e scompaiono quando la sessione termina.

> 💡 **Suggerimento:** potresti trovare notebook che creano file con il comando `%%writefile nome_del_file` all'inizio della cella. Non è Python: è un "comando magico" di Jupyter/Colab. Qui usiamo Python puro perché il codice funzioni in qualsiasi ambiente.

### 1.3 Aprire file con `with` e `open()`

Per lavorare con un file, prima dobbiamo **aprirlo**. In Python, lo facciamo con la funzione nativa `open()`, quasi sempre accompagnata dalla parola chiave `with`:

```python
with open('percorso/del/file', mode='modo', encoding='utf-8') as alias:
    # blocco di codice che usa il file (indentato)
```

| Parte | Cosa significa |
| --- | --- |
| `'percorso/del/file'` | Nome (o percorso) del file. Senza cartella, Python cerca nella cartella corrente. |
| `mode` | Modalità di apertura. `'r'` (*read*) è lettura ed è il valore predefinito. Vedremo le altre nella sezione 2. |
| `encoding` | Codifica del testo. Usa `'utf-8'` affinché accenti e caratteri speciali funzionino correttamente. |
| `as alias` | Nome della variabile che rappresenta il file aperto all'interno del blocco. |

Il `with` garantisce che il file venga **chiuso automaticamente** alla fine del blocco, anche se si verifica un errore nel mezzo. Un file dimenticato aperto può restare bloccato, consumare memoria o addirittura perdere dati non ancora salvati.

> ⚠️ **Attenzione:** se il file non esiste, `open()` in modalità lettura genera un errore `FileNotFoundError`. Possiamo gestire questa situazione con il `try/except` visto nel modulo precedente:

In [2]:
# Tentativo di aprire un file che non esiste
try:
    with open('arquivo_que_nao_existe.csv', mode='r', encoding='utf-8') as arquivo:
        print(arquivo.read())
except FileNotFoundError:
    print('File non trovato. Controlla il nome e la cartella.')

Arquivo não encontrado. Confira o nome e a pasta.


### 1.4 Leggere tutto con `read()`

Il metodo `.read()` legge **tutto il contenuto** del file in una sola volta e lo restituisce come un'unica *stringa*:

In [3]:
with open('banco.csv', mode='r', encoding='utf-8') as arquivo:
    conteudo = arquivo.read()

# Fuori dal blocco, il file è già stato chiuso, ma il testo letto resta nella variabile
print(conteudo)
print(type(conteudo))

age,job,marital,education,default,balance,housing,loan
30,unemployed,married,primary,no,1787,no,no
33,services,married,secondary,no,4789,yes,yes
35,management,single,tertiary,no,1350,yes,no
30,management,married,tertiary,no,1476,yes,yes
59,blue-collar,married,secondary,no,0,yes,no
35,management,single,tertiary,no,747,no,no
36,self-employed,married,tertiary,no,307,yes,no
39,technician,married,secondary,no,147,yes,no
41,entrepreneur,married,tertiary,no,221,yes,no
43,services,married,primary,no,-88,yes,yes

<class 'str'>


> 💡 **Suggerimento:** in modalità `'r'` il file può solo essere letto. Qualsiasi tentativo di scriverci genera un errore, il che protegge il file originale da modifiche accidentali.

### 1.5 Leggere riga per riga

Il `.read()` è pratico, ma carica l'intero file in memoria. Per file grandi (centinaia di *megabyte* o più), l'ideale è leggere **una riga alla volta**. Python offre tre modi:

| Forma | Cosa restituisce | Quando usarla |
| --- | --- | --- |
| `arquivo.readline()` | La **riga successiva** del file, come *stringa* | Per leggere una riga specifica, come l'intestazione |
| `for linha in arquivo:` | Una riga a ogni ripetizione del ciclo | Per scorrere l'intero file senza caricarlo tutto in memoria |
| `arquivo.readlines()` | Una **lista** con tutte le righe | File piccoli, quando servono le righe in una lista |

**Esempio:** il `.readline()` legge una riga e "avanza" nel file, quindi la chiamata successiva restituisce già la riga seguente.

In [4]:
with open('banco.csv', mode='r', encoding='utf-8') as arquivo:
    cabecalho = arquivo.readline()        # 1ª riga: l'intestazione
    primeiro_cliente = arquivo.readline()  # 2ª riga: il primo cliente

print(cabecalho)
print(primeiro_cliente)

age,job,marital,education,default,balance,housing,loan

30,unemployed,married,primary,no,1787,no,no



Nota che è apparsa una **riga vuota** dopo ogni testo. Questo accade perché ogni riga letta termina con il carattere invisibile `\n` (a capo), e il `print()` ne aggiunge un altro. La funzione `repr()` mostra la *stringa* "grezza", con questi caratteri visibili:

In [5]:
# repr() rivela i caratteri invisibili, come il \n alla fine della riga
print(repr(cabecalho))

# .strip() rimuove spazi e a capo dall'inizio e dalla fine
print(repr(cabecalho.strip()))

'age,job,marital,education,default,balance,housing,loan\n'
'age,job,marital,education,default,balance,housing,loan'


**Esempio:** scorrendo l'intero file con `for`, che è il modo più semplice ed efficiente per leggere riga per riga.

In [6]:
with open('banco.csv', mode='r', encoding='utf-8') as arquivo:
    for linha in arquivo:
        # .strip() evita la riga vuota extra tra i record
        print(linha.strip())

age,job,marital,education,default,balance,housing,loan
30,unemployed,married,primary,no,1787,no,no
33,services,married,secondary,no,4789,yes,yes
35,management,single,tertiary,no,1350,yes,no
30,management,married,tertiary,no,1476,yes,yes
59,blue-collar,married,secondary,no,0,yes,no
35,management,single,tertiary,no,747,no,no
36,self-employed,married,tertiary,no,307,yes,no
39,technician,married,secondary,no,147,yes,no
41,entrepreneur,married,tertiary,no,221,yes,no
43,services,married,primary,no,-88,yes,yes


**Esempio:** il `.readlines()` restituisce tutte le righe in una lista, ognuna con ancora il `\n` alla fine.

In [7]:
with open('banco.csv', mode='r', encoding='utf-8') as arquivo:
    linhas = arquivo.readlines()

print(len(linhas))  # intestazione + 10 clienti
print(linhas[:3])   # le prime tre righe

11
['age,job,marital,education,default,balance,housing,loan\n', '30,unemployed,married,primary,no,1787,no,no\n', '33,services,married,secondary,no,4789,yes,yes\n']


> ⚠️ **Attenzione:** quando il file arriva alla fine, `.readline()` restituisce una *stringa* **vuota** (`''`), e non `None`. Potresti trovare codici che usano un ciclo `while` con `.readline()` per leggere il file, fermandosi quando la *stringa* arriva vuota. Il `for linha in arquivo` fa esattamente la stessa cosa, con meno codice e meno probabilità di errore.

### 1.6 Rivisitando la motivazione

Ora possiamo calcolare l'età media dei clienti. Il piano è:

1. Leggere e scartare l'intestazione con `.readline()`;
2. Scorrere le righe restanti con `for`;
3. In ogni riga, rimuovere il `\n` con `.strip()` e separare le colonne con `.split(',')`;
4. Prendere la prima colonna (l'età), convertirla in `int` e salvarla in una lista.

In [8]:
idades = []

with open('banco.csv', mode='r', encoding='utf-8') as arquivo:
    arquivo.readline()  # scarta l'intestazione

    for linha in arquivo:
        colunas = linha.strip().split(',')  # '30,unemployed,...' -> ['30', 'unemployed', ...]
        idade = int(colunas[0])             # l'età è la 1ª colonna e arriva come testo
        idades.append(idade)

print(idades)

[30, 33, 35, 30, 59, 35, 36, 39, 41, 43]


Con le età in una lista, la media è la somma divisa per il numero di elementi:

In [9]:
idade_media = sum(idades) / len(idades)
print(f'Età media dei clienti: {idade_media:.1f} anni')

Idade média dos clientes: 38.1 anos


> ⚠️ **Attenzione:** tutto ciò che viene letto da un file di testo arriva come `str`, anche se sembra un numero. Senza `int()`, la lista avrebbe `'30'`, `'33'` e così via, e `sum()` genererebbe un errore.

> 💡 **Suggerimento:** separare le colonne con `.split(',')` funziona bene per file semplici, ma fallisce quando un valore contiene una virgola (per esempio, `"São Paulo, SP"`). Per questi casi esistono il modulo `csv` di Python e la libreria **Pandas**, che vedremo nei moduli futuri.

---

## 2. Scrittura di file

Tanto importante quanto leggere dati è **salvare i risultati**: un report, una lista filtrata, un file per un altro sistema. In questa sezione creeremo file, li sovrascriveremo e aggiungeremo contenuto ad essi.

### 2.1 Modalità di apertura

Il parametro `mode` di `open()` definisce cosa possiamo fare con il file:

| Modalità | Nome | Se il file esiste... | Se il file non esiste... |
| --- | --- | --- | --- |
| `'r'` | Lettura (*read*) | Apre in lettura | Genera `FileNotFoundError` |
| `'w'` | Scrittura (*write*) | **Cancella tutto il contenuto** e scrive da zero | Crea il file |
| `'a'` | Aggiunta (*append*) | Scrive **alla fine**, mantenendo il contenuto | Crea il file |
| `'x'` | Creazione esclusiva | Genera `FileExistsError` | Crea il file |

> ⚠️ **Attenzione:** la modalità `'w'` cancella il file senza chiedere conferma. Se l'obiettivo è aggiungere contenuto, usa `'a'`. Se vuoi essere sicuro di non sovrascrivere nulla, usa `'x'`.

### 2.2 Scrivere con `write()`

Il metodo `.write()` scrive una *stringa* nel file. **Non** aggiunge automaticamente l'a capo, quindi dobbiamo includere il `\n` alla fine di ogni riga.

**Esempio:** salvando le età estratte nella sezione 1 in un nuovo file, `idades.csv`.

In [10]:
with open('idades.csv', mode='w', encoding='utf-8') as arquivo:
    arquivo.write('idade\n')  # intestazione

    for idade in idades:
        # write() accetta solo str: convertiamo il numero e aggiungiamo l'a capo
        arquivo.write(f'{idade}\n')

print('File idades.csv salvato!')

Arquivo idades.csv gravado!


Leggiamo di nuovo il file per verificare cosa è stato scritto:

In [11]:
with open('idades.csv', mode='r', encoding='utf-8') as arquivo:
    print(arquivo.read())

idade
30
33
35
30
59
35
36
39
41
43



> ⚠️ **Attenzione:** passare un numero direttamente a `.write()`, come `arquivo.write(30)`, genera un `TypeError`. Convertilo con `str()` o usa una *f-string*, come abbiamo fatto sopra.

### 2.3 Aggiungere contenuto con la modalità `'a'`

**Esempio:** sono stati registrati due nuovi clienti e dobbiamo includere le loro età alla fine del file, senza perdere quelle già presenti.

In [12]:
novas_idades = [25, 48]

# Modalità 'a': il contenuto esistente viene mantenuto e quello nuovo va alla fine
with open('idades.csv', mode='a', encoding='utf-8') as arquivo:
    for idade in novas_idades:
        arquivo.write(f'{idade}\n')

with open('idades.csv', mode='r', encoding='utf-8') as arquivo:
    print(arquivo.read())

idade
30
33
35
30
59
35
36
39
41
43
25
48



> 💡 **Suggerimento:** se esegui la cella sopra più di una volta, le età `25` e `48` verranno aggiunte di nuovo a ogni esecuzione. Per tornare allo stato iniziale, esegui di nuovo la cella della sezione 2.2, che usa la modalità `'w'` e ricrea il file da zero.

### 2.4 Leggere da un file e scrivere in un altro

È molto comune leggere un file, elaborare le righe e salvare il risultato in un altro. Un unico `with` può aprire entrambi i file, separati da virgola.

**Esempio:** copiando il `banco.csv` in un file con un'altra estensione, `banco.txt`.

In [13]:
with open('banco.csv', mode='r', encoding='utf-8') as leitura, \
     open('banco.txt', mode='w', encoding='utf-8') as escrita:
    for linha in leitura:
        escrita.write(linha)  # la riga termina già con \n

print('Copia completata!')

Cópia concluída!


> 💡 **Suggerimento:** l'estensione (`.csv`, `.txt`) è solo parte del nome del file e serve a indicare ai programmi come aprirlo. Il contenuto dei due file è esattamente lo stesso testo.

**Esempio:** il responsabile crediti ha chiesto un file con solo i clienti che hanno un **prestito personale** (colonna `loan`, l'8ª colonna, con valore `yes`).

In [14]:
with open('banco.csv', mode='r', encoding='utf-8') as leitura, \
     open('clientes_com_emprestimo.csv', mode='w', encoding='utf-8') as escrita:
    # L'intestazione viene copiata così com'è
    escrita.write(leitura.readline())

    for linha in leitura:
        colunas = linha.strip().split(',')
        tem_emprestimo = colunas[7] == 'yes'  # indice 7 = 8ª colonna (loan)
        if tem_emprestimo:
            escrita.write(linha)

with open('clientes_com_emprestimo.csv', mode='r', encoding='utf-8') as arquivo:
    print(arquivo.read())

age,job,marital,education,default,balance,housing,loan
33,services,married,secondary,no,4789,yes,yes
30,management,married,tertiary,no,1476,yes,yes
43,services,married,primary,no,-88,yes,yes



---

## 3. Funzioni

Man mano che i programmi crescono, gli stessi pezzi di codice iniziano a ripetersi. Le **funzioni** permettono di dare un nome a un blocco di codice ed eseguirlo ogni volta che serve, con valori diversi ogni volta. Questo rende il codice più breve, più organizzato e più facile da correggere.

### 3.1 Motivazione

Lavori in una società di intermediazione finanziaria e devi simulare il rendimento di investimenti con **interesse composto** in vari scenari. Nel regime a interesse composto, ogni anno il rendimento viene calcolato sul valore accumulato fino a quel momento:

$$\text{valore alla fine dell'anno} = \text{valore all'inizio dell'anno} \times (1 + \text{tasso di interesse annuo})$$

Ecco come appare il codice per simulare solo **due** scenari:

In [15]:
# Scenario 1
valor_inicial = 1000.00
taxa_juros_anual = 0.05
anos = 10

valor_final = valor_inicial
for ano in range(anos):
    valor_final = valor_final * (1 + taxa_juros_anual)

print(f'R$ {valor_inicial:.2f} al {taxa_juros_anual:.0%} annuo, in {anos} anni: R$ {valor_final:.2f}')

# Scenario 2: esattamente lo stesso codice, cambiano solo i valori
valor_inicial = 1020.00
taxa_juros_anual = 0.03
anos = 10

valor_final = valor_inicial
for ano in range(anos):
    valor_final = valor_final * (1 + taxa_juros_anual)

print(f'R$ {valor_inicial:.2f} al {taxa_juros_anual:.0%} annuo, in {anos} anni: R$ {valor_final:.2f}')

R$ 1000.00 a 5% ao ano, em 10 anos: R$ 1628.89
R$ 1020.00 a 3% ao ano, em 10 anos: R$ 1370.79


> 💡 **Suggerimento:** nella *f-string*, il formato `:.0%` moltiplica il numero per 100 e aggiunge il simbolo di percentuale: `0.05` diventa `5%`.

E se fossero 50 scenari? Copiare e incollare lo stesso blocco 50 volte renderebbe il codice enorme, e un errore nel calcolo dovrebbe essere corretto in 50 punti.

**Come possiamo riutilizzare questo codice ed evitare tanta ripetizione?**

### 3.2 Definizione

Una **funzione** è un blocco di codice con un nome, che **viene eseguito solo quando viene chiamato**. Abbiamo già usato diverse funzioni pronte di Python, come `print()`, `len()` e `open()`. Ora creeremo le nostre.

**Definire** una funzione:

```python
def nome_della_funzione(parametro_1, parametro_2):
    # blocco di codice (indentato)
    return valore_di_ritorno
```

**Chiamare** una funzione:

```python
risultato = nome_della_funzione(argomento_1, argomento_2)
```

| Parte | Cosa significa |
| --- | --- |
| `def` | Parola chiave che avvia la definizione di una funzione |
| `nome_della_funzione` | Nome della funzione, in `snake_case` e preferibilmente con un verbo (`calcular_`, `ler_`, `exibir_`) |
| `(parametro_1, ...)` | **Parametri**: variabili che ricevono i valori inviati nella chiamata |
| `return` | Restituisce un risultato a chi ha chiamato la funzione |

**Esempio:** una funzione che mostra un messaggio a schermo.

In [16]:
# Definire la funzione NON esegue il codice: lo salva soltanto con un nome
def exibir_mensagem(mensagem):
    print(mensagem)

In [17]:
# Ora sì, la funzione viene eseguita (e può essere chiamata tutte le volte che vogliamo)
exibir_mensagem('Ciao a tutti! Benvenuti al modulo 04.')
exibir_mensagem('Le funzioni evitano la ripetizione del codice.')

Olá, pessoal! Bem-vindos ao módulo 04.
Funções evitam repetição de código.


> ⚠️ **Attenzione:** come le variabili, la funzione deve essere **definita prima** di essere chiamata. Se chiami `exibir_mensagem()` senza aver eseguito la cella con il `def`, Python genererà un `NameError`.

> 💡 **Suggerimento:** è buona pratica descrivere cosa fa la funzione in una ***docstring***, un testo tra triple virgolette subito dopo il `def`. Appare quando qualcuno usa `help()` sulla funzione.

In [18]:
def exibir_mensagem(mensagem):
    '''Mostra a schermo il messaggio ricevuto.'''
    print(mensagem)

help(exibir_mensagem)

Help on function exibir_mensagem in module __main__:

exibir_mensagem(mensagem)
    Exibe a mensagem recebida na tela.



### 3.3 Parametri e argomenti

I **parametri** sono le variabili dichiarate tra parentesi nel `def`. Gli **argomenti** sono i valori che inviamo a questi parametri al momento della chiamata. Una funzione può avere nessuno, uno o più parametri.

**Esempio:** funzione **senza parametri**.

In [19]:
def obter_pi():
    return 3.14159265359

valor_pi = obter_pi()
print(valor_pi)

3.14159265359


> ⚠️ **Attenzione:** non salvare mai il risultato in una variabile con lo **stesso nome** della funzione, come `obter_pi = obter_pi()`. La variabile sostituisce la funzione, e una seconda chiamata `obter_pi()` genera un `TypeError`, perché ora il nome punta a un `float`, e non più a una funzione.

**Esempio:** funzione **con più parametri**. Gli argomenti possono essere passati in due modi:

- **Posizionali:** nello stesso ordine dei parametri;
- **Nominati:** indicando `parametro=valore`. In questo caso l'ordine non conta, e la chiamata risulta più leggibile.

In [20]:
def calcular_total_compra(preco_unitario, quantidade):
    return preco_unitario * quantidade

# Argomenti posizionali: l'ordine conta
print(calcular_total_compra(12.50, 4))

# Argomenti nominati: l'ordine non conta
print(calcular_total_compra(quantidade=4, preco_unitario=12.50))

50.0
50.0


**Esempio:** parametri con **valore predefinito**. Se l'argomento non viene indicato nella chiamata, il parametro assume il valore definito nel `def`.

In [21]:
def calcular_preco_final(preco, desconto=0.0):
    return preco * (1 - desconto)

print(calcular_preco_final(100))                # usa lo sconto predefinito (0%)
print(calcular_preco_final(100, desconto=0.15))  # sconto del 15%

100.0
85.0


> ⚠️ **Attenzione:** i parametri con valore predefinito devono venire **dopo** i parametri obbligatori. Una definizione come `def calcular_preco_final(desconto=0.0, preco):` genera un `SyntaxError`.

Possiamo anche indicare il **tipo atteso** di ogni parametro e del valore di ritorno, usando le ***type hints*** (indicazioni di tipo): `parametro: tipo` e `-> tipo_del_ritorno`.

In [22]:
def calcular_preco_final(preco: float, desconto: float = 0.0) -> float:
    return preco * (1 - desconto)

print(calcular_preco_final(250.0, desconto=0.10))

225.0


> 💡 **Suggerimento:** le *type hints* servono come documentazione e aiutano gli editor di codice a segnalare errori, ma Python **non** le verifica durante l'esecuzione. Una chiamata con un tipo diverso da quello indicato continua a funzionare (o a fallire) normalmente.

### 3.4 Ritorno (return)

Il `return` restituisce un valore a chi ha chiamato la funzione. Questo valore può essere salvato in una variabile, stampato o usato in altri calcoli.

In [23]:
def converter_para_maiusculas(texto: str) -> str:
    texto_maiusculo = texto.upper()
    return texto_maiusculo

nome = 'André Perez'
nome_maiusculo = converter_para_maiusculas(nome)

print(nome)
print(nome_maiusculo)

André Perez
ANDRÉ PEREZ


**Mostrare non è la stessa cosa di restituire.** Una funzione senza `return` restituisce automaticamente il valore `None`. Per questo, il risultato di una funzione che usa solo `print()` non può essere sfruttato in seguito:

In [24]:
def exibir_dobro(numero):
    print(numero * 2)   # mostra soltanto a schermo

def calcular_dobro(numero):
    return numero * 2   # restituisce il valore

resultado_exibir = exibir_dobro(5)
resultado_calcular = calcular_dobro(5)

print(f'exibir_dobro ha restituito:  {resultado_exibir}')
print(f'calcular_dobro ha restituito: {resultado_calcular}')

10
exibir_dobro devolveu:  None
calcular_dobro devolveu: 10


Il `return` **termina anche la funzione** immediatamente: nulla di ciò che viene dopo, nello stesso percorso, viene eseguito. Questo permette di restituire valori diversi in base a una condizione.

**Esempio:** classificando il saldo di un cliente della banca.

In [25]:
def classificar_saldo(saldo: float) -> str:
    if saldo < 0:
        return 'negativo'   # la funzione termina qui se il saldo è negativo
    if saldo == 0:
        return 'zero'
    return 'positivo'

print(classificar_saldo(-88))
print(classificar_saldo(0))
print(classificar_saldo(1787))

negativo
zerado
positivo


Una funzione può **restituire più di un valore**, separandoli con la virgola. In realtà, Python raggruppa i valori in una **tupla**, che possiamo "scompattare" in più variabili nella chiamata.

**Esempio:** separando l'utente e il provider di un'email.

In [26]:
def extrair_usuario_e_provedor(email: str) -> tuple[str, str]:
    partes = email.split('@')
    usuario = partes[0]
    provedor = partes[1]
    return usuario, provedor

usuario, provedor = extrair_usuario_e_provedor('andre.perez@gmail.com')

print(usuario)
print(provedor)

andre.perez
gmail.com


### 3.5 Gestire gli errori dentro le funzioni

Le funzioni che gestiscono file possono fallire per vari motivi: cartella inesistente, mancanza di permessi, dati in un formato inaspettato. Con `try/except`, la funzione può gestire l'errore e **informare** chi l'ha chiamata se l'operazione è andata a buon fine.

**Esempio:** una funzione riutilizzabile per salvare un file CSV di una colonna, che restituisce `True` in caso di successo e `False` in caso di errore.

In [27]:
def escrever_arquivo_csv(nome_arquivo: str, cabecalho: str, valores: list) -> bool:
    '''Salva un CSV di una colonna. Restituisce True se è andato a buon fine e False in caso di errore.'''
    try:
        with open(nome_arquivo, mode='w', encoding='utf-8') as arquivo:
            arquivo.write(f'{cabecalho}\n')
            for valor in valores:
                arquivo.write(f'{valor}\n')
    except Exception as erro:
        print(f'Non è stato possibile salvare il file: {erro}')
        return False

    return True

In [28]:
# Chiamata corretta: valores è una lista
gravou = escrever_arquivo_csv('idades_funcao.csv', 'idade', [30, 33, 35, 30, 59])
print(f'Salvato con successo? {gravou}')

Gravou com sucesso? True


In [29]:
# Chiamata con errore: 10 non è una lista, quindi il for dentro la funzione fallisce
gravou = escrever_arquivo_csv('idades_funcao_erro.csv', 'idade', 10)
print(f'Salvato con successo? {gravou}')

Não foi possível gravar o arquivo: 'int' object is not iterable
Gravou com sucesso? False


> 💡 **Suggerimento:** nota che il programma **non si è bloccato** nel secondo caso. L'errore è stato gestito dentro la funzione, e il valore `False` permette al resto del codice di decidere cosa fare.

### 3.6 Rivisitando la motivazione

Ora possiamo mettere il calcolo dell'interesse composto in una funzione. Nota che essa si limita a **calcolare e restituire** il valore. Mostrare il risultato spetta a chi la chiama, il che rende la funzione più facile da riutilizzare (per esempio, per salvare i risultati in un file invece di stamparli).

In [30]:
def calcular_juros_compostos(valor_inicial: float, taxa_juros_anual: float, anos: int) -> float:
    '''Restituisce il valore finale di un investimento con interesse composto annuo.'''
    valor_final = valor_inicial
    for _ in range(anos):  # "_" indica che la variabile del ciclo non viene usata
        valor_final = valor_final * (1 + taxa_juros_anual)
    return valor_final

Gli stessi due scenari di prima, ora con una riga ciascuno:

In [31]:
valor_final_1 = calcular_juros_compostos(valor_inicial=1000.00, taxa_juros_anual=0.05, anos=10)
valor_final_2 = calcular_juros_compostos(valor_inicial=1020.00, taxa_juros_anual=0.03, anos=10)

print(f'Scenario 1: R$ {valor_final_1:.2f}')
print(f'Scenario 2: R$ {valor_final_2:.2f}')

Cenário 1: R$ 1628.89
Cenário 2: R$ 1370.79


E simulare vari scenari diventa banale. **Esempio:** confrontando diversi tassi per lo stesso valore iniziale.

In [32]:
taxas_simuladas = [0.03, 0.05, 0.08, 0.10]

for taxa in taxas_simuladas:
    valor_final = calcular_juros_compostos(valor_inicial=1000.00, taxa_juros_anual=taxa, anos=10)
    print(f'Tasso del {taxa:.0%} annuo: R$ {valor_final:.2f}')

Taxa de 3% ao ano: R$ 1343.92
Taxa de 5% ao ano: R$ 1628.89
Taxa de 8% ao ano: R$ 2158.92
Taxa de 10% ao ano: R$ 2593.74


> 💡 **Suggerimento:** l'interesse composto ha anche una formula diretta, $\text{valore finale} = \text{valore iniziale} \times (1 + \text{tasso})^{\text{anni}}$, che in Python diventa `valor_inicial * (1 + taxa_juros_anual) ** anos`. Dato che il calcolo è isolato in una funzione, potremmo sostituire il ciclo con la formula **senza cambiare nessuna chiamata**. Questo è un altro grande vantaggio delle funzioni.

In [33]:
# Verifica: la formula diretta dà lo stesso risultato del ciclo
print(round(1000.00 * (1 + 0.05) ** 10, 2))
print(round(calcular_juros_compostos(1000.00, 0.05, 10), 2))

1628.89
1628.89


> ⚠️ **Attenzione:** il `round()` di Python usa l'**arrotondamento bancario**: quando il numero è esattamente a metà (`.5`), arrotonda al numero **pari** più vicino. Per questo `round(2.5)` dà come risultato `2`, e non `3`. Nella maggior parte dei casi questo non fa differenza, ma è bene saperlo per non sorprendersi.

In [34]:
print(round(10 / 4))  # 2.5 -> 2 (il pari più vicino)
print(round(3.5))     # 3.5 -> 4 (il pari più vicino)
print(round(2.6))     # 2.6 -> 3 (non è a metà: arrotonda normalmente)

2
4
3


---

## 4. Scope

Con le funzioni, sorge una domanda importante: una variabile creata dentro una funzione esiste anche fuori di essa? E viceversa? Lo **scope** (ambito di visibilità) definisce **dove** una variabile può essere usata e **per quanto tempo** esiste. Capire questo evita errori difficili da trovare.

### 4.1 Definizione

Lo **scope** di una variabile è la regione di codice in cui è visibile. In Python, i due scope più importanti sono:

| Scope | Dove viene creata la variabile | Dove può essere usata | Per quanto tempo esiste |
| --- | --- | --- | --- |
| **Locale** | Dentro una funzione (inclusi i parametri) | Solo dentro quella funzione | Finché la funzione è in esecuzione |
| **Globale** | Fuori da qualsiasi funzione (nel "corpo" del notebook) | Ovunque, anche dentro le funzioni (in lettura) | Fino a quando il programma (o la sessione) termina |

### 4.2 Scope locale

Le variabili create dentro una funzione sono **locali**: nascono quando la funzione viene chiamata e vengono eliminate quando essa termina.

**Esempio:** sommando i valori di una lista.

In [35]:
def somar_lista(numeros: list) -> int:
    total = 0  # variabile locale: esiste solo dentro la funzione
    for numero in numeros:
        total = total + numero
    return total

soma = somar_lista([2, 4, 6, 8])
print(soma)

20


Fuori dalla funzione, la variabile `total` **non esiste**. Provare a usarla genera un errore:

In [36]:
# print(total)    # ❌ NameError: name 'total' is not defined

# ✅ Il risultato arriva qui fuori tramite il return, salvato nella variabile soma
print(soma)

20


> 💡 **Suggerimento:** Python ha già la funzione nativa `sum()`, che fa esattamente ciò che fa `somar_lista()`. Abbiamo creato la nostra solo per capire lo scope, ma nella pratica preferisci le funzioni native.

### 4.3 Scope globale

Le variabili create fuori dalle funzioni sono **globali** e possono essere **lette** dentro qualsiasi funzione:

In [37]:
taxa_administracao = 0.02  # variabile globale

def calcular_valor_liquido(valor_bruto: float) -> float:
    # La funzione riesce a LEGGERE la variabile globale
    return valor_bruto * (1 - taxa_administracao)

print(calcular_valor_liquido(1000))

980.0


Attenzione però: se la funzione **assegna** un valore a una variabile con lo stesso nome, Python crea una **nuova variabile locale**, e quella globale resta intatta:

In [38]:
def simular_taxa_maior(valor_bruto: float) -> float:
    taxa_administracao = 0.10  # crea una variabile LOCALE con lo stesso nome
    return valor_bruto * (1 - taxa_administracao)

print(simular_taxa_maior(1000))  # usa il tasso locale (10%)
print(taxa_administracao)        # quella globale resta 0.02

900.0
0.02


Per **modificare** una variabile globale dentro una funzione, è necessario dichiararla con la parola chiave `global`:

In [39]:
contador_simulacoes = 0

def registrar_simulacao():
    # contador_simulacoes += 1  # ❌ senza "global": UnboundLocalError
    global contador_simulacoes
    contador_simulacoes += 1

registrar_simulacao()
registrar_simulacao()
print(contador_simulacoes)

2


> ⚠️ **Attenzione:** usare `global` rende il codice difficile da capire e testare, perché la funzione finisce per dipendere da qualcosa di "nascosto" fuori da essa. Preferisci la strada più chiara: **ricevere i dati tramite parametri e restituire il risultato con `return`**.

In [40]:
# ✅ Stesso risultato, senza global: la funzione riceve il valore e ne restituisce uno nuovo
def incrementar(contador: int) -> int:
    return contador + 1

contador = 0
contador = incrementar(contador)
contador = incrementar(contador)
print(contador)

2


### 4.4 Strutture condizionali e di ripetizione

A differenza di altri linguaggi, in Python `if` e `for` **non creano un proprio scope**. Una variabile creata al loro interno continua a esistere dopo la fine del blocco.

In [41]:
saldo = 1787

if saldo > 0:
    situacao = 'positivo'  # creata dentro l'if...
else:
    situacao = 'negativo'

print(situacao)  # ...ma accessibile fuori da esso

positivo


In [42]:
for indice in range(3):
    ultimo_processado = indice

# Dopo il ciclo, entrambe le variabili continuano a esistere con l'ultimo valore
print(indice)
print(ultimo_processado)

2
2


> ⚠️ **Attenzione:** una variabile creata in un blocco che **non è stato eseguito** semplicemente non esiste. Se nell'esempio sopra solo l'`if` definisse `situacao` (senza l'`else`) e il saldo fosse negativo, `print(situacao)` genererebbe un `NameError`. Per questo, assicurati che la variabile riceva un valore in tutti i percorsi possibili, oppure creala con un valore iniziale prima dell'`if`.

In [43]:
saldo = -88

# ✅ Valore iniziale definito prima dell'if: la variabile esiste sempre
situacao = 'non negativo'
if saldo < 0:
    situacao = 'negativo'

print(situacao)

negativo


---

## Riepilogo del Modulo

| Concetto | A cosa serve | Esempio |
| --- | --- | --- |
| `with open(...) as arquivo:` | Apre un file e lo chiude automaticamente alla fine del blocco | `with open('banco.csv', mode='r', encoding='utf-8') as arquivo:` |
| `.read()` | Legge l'intero file come una *stringa* | `conteudo = arquivo.read()` |
| `.readline()` | Legge la riga successiva | `cabecalho = arquivo.readline()` |
| `for linha in arquivo:` | Scorre il file riga per riga | `for linha in arquivo: ...` |
| `.readlines()` | Legge tutte le righe in una lista | `linhas = arquivo.readlines()` |
| `.write()` | Scrive una *stringa* (senza a capo automatico) | `arquivo.write(f'{idade}\n')` |
| Modalità `'r'`, `'w'`, `'a'`, `'x'` | Lettura, scrittura (cancella), aggiunta e creazione esclusiva | `open('idades.csv', mode='a')` |
| `def` | Definisce una funzione | `def calcular_dobro(numero):` |
| Parametro con valore predefinito | Rende l'argomento opzionale | `def calcular_preco_final(preco, desconto=0.0):` |
| `return` | Restituisce un risultato e termina la funzione | `return numero * 2` |
| Scope locale | La variabile creata nella funzione esiste solo al suo interno | `total` in `somar_lista()` |
| Scope globale | La variabile creata fuori dalle funzioni può essere letta ovunque | `taxa_administracao` |

Funzioni e metodi visti in questo modulo: `open()`, `.read()`, `.readline()`, `.readlines()`, `.write()`, `.strip()`, `.split()`, `repr()`, `help()`, `sum()` e `round()`.